[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/takeshun1984/NumeralAnalysisInGeophysics_SolidEarth/blob/main/07_FDM_2nd_homo_P-SV_CerjanGQ.ipynb)

In [1]:
import numpy as np
import os

# 領域設定など
SP = np.float32
NX, NZ = 400, 400
DX, DZ = 0.4, 0.4
DT = 0.02
NTMAX = 2000

# 震源情報
I0, K0 = NX // 2, NZ // 2
T0, TS = 5.0, 0.0
MXX, MZZ, MXZ, MO = 0.0, 0.0, 1.0, 1.0
DTXZ = DT / (DX * DZ)

# --- 震源情報 ---
I0, K0 = NX // 2, NZ // 2
T0, TS = 5.0, 0.0
MXX, MZZ, MXZ, MO = 0.0, 0.0, 1.0, 1.0

# --- 出力・減衰設定 ---
NTDEC, NXD, NZD = 50, 2, 2
ONAME0, WNAME0 = "output/psv.h.", "output/wav.h."

# --- 関数定義 ---
def kupper(t, ts, tr):
    if ts <= t <= ts + tr:
        return 3 * np.pi * (np.sin(np.pi * (t - ts) / tr))**3 / (4 * tr)
    else:
        return 0.0

# 配列の初期化 0~NX+1
SXX = np.zeros((NX + 2, NZ + 2), dtype=SP)
SZZ = np.zeros((NX + 2, NZ + 2), dtype=SP)
SXZ = np.zeros((NX + 2, NZ + 2), dtype=SP)
VX  = np.zeros((NX + 2, NZ + 2), dtype=SP)
VZ  = np.zeros((NX + 2, NZ + 2), dtype=SP)

# 均質媒質
VP, VS, RO = 6.0, 3.5, 2.3
RIG_val = RO * VS**2
LAM_val = RO * VP**2 - 2.0 * RO * VS**2

# 各格子点に物性を割り当て
RIG = np.full((NX + 2, NZ + 2), RIG_val, dtype=SP)
LAM = np.full((NX + 2, NZ + 2), LAM_val, dtype=SP)
RHO = np.full((NX + 2, NZ + 2), RO, dtype=SP)

# --- 吸収境界条件 (Cerjan 1985) パラメータ ---
BETA, NA = 0.09, 20

# --- 吸収境界係数の計算 ---
gx1, gx2 = np.ones(NX+2, dtype=SP), np.ones(NX+2, dtype=SP)
gz1, gz2 = np.ones(NZ+2, dtype=SP), np.ones(NZ+2, dtype=SP)
for i in range(1, NA + 1):
    val1 = np.exp(-BETA*(1.0-(i-0.5)/NA)**2)
    val2 = np.exp(-BETA*(1.0-i/NA)**2)
    
    gx1[i], gz1[i], gx2[i], gz2[i] = val1, val1, val2, val2

    gx1[NX-i+1], gx2[NX-i+1], gz1[NZ-i+1], gz2[NZ-i+1] = val2, val1, val2, val1

GX1, GZ1 = np.meshgrid(gx1, gz1, indexing='ij')
GX2, GZ2 = np.meshgrid(gx2, gz2, indexing='ij')

ここまでは、前回の吸収境界のみコードとの共通部分。

本コードではQをGraves 1996の提案の形式でまずは挿入します。
$$
v_p \gets v_p \exp{\left[-\frac{\pi f_0 \Delta t}{Q}\right]}
$$
とする。この手法は、$\Delta t$がかかっているので、$n$タイムステップ（$t = n\Delta t$）計算すると、
$$
\left(\exp{\left[-\frac{\pi f_0 \Delta t}{Q}\right]}\right)^n =\exp{\left[-\frac{\pi f_0 n\Delta t}{Q}\right]} =\exp{\left[-\frac{\pi f_0 t}{Q}\right]}
$$
と、時刻$t$においては均質媒質からの差が、上式のように書ける。

In [2]:
F0, Q0 = 1.0 / T0, 10.0

# 減衰（QI）の設定
QI = np.ones((NX+2, NZ+2), dtype=SP)
QI[1 : NX//2 - 1, 1:NZ+1] = np.exp(-np.pi * F0 * DT / Q0)

地球の現実的な$Q_S$は100~400程度なのですが、ここでは効果がすぐにわかるように$Q=10$として強い値を入れています。また、$f_0 = 1/T_0$として、震源から出る地震波の卓越周波数付近としています。
ここで、
```
QI[1 : NX//2 - 1, 1:NZ+1] = np.exp(-np.pi * F0 * DT / Q0)
```
としていますが、1~NX/2-2まで強い減衰を含めています。これは震源が中央にあるため、震源を均質にするためにそのようにしています。


また、波形の出力を、$NZ/2$の計算領域を上下に分ける中心部で、$X$方向へ10 km毎に設置する。すると、観測点数は$NX*\Delta x = 160$ kmですので、15点となります。波形保存用のVWXとVWZを初期化して、観測点位置のグリッドをISTXとISTZで定義しています。

In [3]:
DST = 10.0
NST = int(NX * DX / DST) - 1
NSKIP = 2
NWMAX = NTMAX // NSKIP

# 波形記録用配列
VWX = np.zeros((NWMAX, NST), dtype=SP)
VWZ = np.zeros((NWMAX, NST), dtype=SP)
ISTX = np.array([int((ist+1)*DST/DX) for ist in range(NST)])
ISTZ = NZ // 2

以下は、メインループで、速度場の更新後に、
```
    VX *= GX2 * GZ1 * QI
    VZ *= GX1 * GZ2 * QI
```
とQIが乗じられていることと、
```
    # 波形記録
    if it % NSKIP == 0:
        it1 = (it // NSKIP) - 1
        if it1 < NWMAX:
            VWX[it1, :] = VX[ISTX, ISTZ]
            VWZ[it1, :] = VZ[ISTX, ISTZ]
```
として、NSKIPに1度、各観測点における波動場を波形としてストアしています。

In [ ]:
# メインループ
print(f"{'Step':>5} / {NTMAX}: {'Time':>7} {'Vxmax':>12}")
os.makedirs("output", exist_ok=True)

for it in range(1, NTMAX + 1):
    T = it * DT

    # 1. 応力場の更新
    dxvx = (VX[1:NX+1, 1:NZ+1] - VX[0:NX, 1:NZ+1]) / DX
    dxvz = (VZ[2:NX+2, 1:NZ+1] - VZ[1:NX+1, 1:NZ+1]) / DX
    dzvx = (VX[1:NX+1, 2:NZ+2] - VX[1:NX+1, 1:NZ+1]) / DZ
    dzvz = (VZ[1:NX+1, 1:NZ+1] - VZ[1:NX+1, 0:NZ]) / DZ

    lam_v, rig_v = LAM[1:NX+1, 1:NZ+1], RIG[1:NX+1, 1:NZ+1]
    SXX[1:NX+1, 1:NZ+1] += ((lam_v + 2.0*rig_v) * dxvx + lam_v * dzvz) * DT
    SZZ[1:NX+1, 1:NZ+1] += ((lam_v + 2.0*rig_v) * dzvz + lam_v * dxvx) * DT
    SXZ[1:NX+1, 1:NZ+1] += rig_v * (dxvz + dzvx) * DT

    SXX *= GX1 * GZ1
    SZZ *= GX1 * GZ1
    SXZ *= GX2 * GZ2

    # 震源注入
    sdrop = MO * kupper(T, TS, T0) * DTXZ
    SXX[I0, K0] -= MXX * sdrop
    SZZ[I0, K0] -= MZZ * sdrop
    SXZ[I0-1:I0+1, K0-1:K0+1] -= MXZ * sdrop * 0.25

    # 2. 速度場の更新
    dxsxx = (SXX[2:NX+2, 1:NZ+1] - SXX[1:NX+1, 1:NZ+1]) / DX
    dxsxz = (SXZ[1:NX+1, 1:NZ+1] - SXZ[0:NX, 1:NZ+1]) / DX
    dzszz = (SZZ[1:NX+1, 2:NZ+2] - SZZ[1:NX+1, 1:NZ+1]) / DZ
    dzsxz = (SXZ[1:NX+1, 1:NZ+1] - SXZ[1:NX+1, 0:NZ]) / DZ

    VX[1:NX+1, 1:NZ+1] += (dxsxx + dzsxz) / RHO[1:NX+1, 1:NZ+1] * DT
    VZ[1:NX+1, 1:NZ+1] += (dxsxz + dzszz) / RHO[1:NX+1, 1:NZ+1] * DT
    
    VX *= GX2 * GZ1 * QI
    VZ *= GX1 * GZ2 * QI

    # 波形記録
    if it % NSKIP == 0:
        it1 = (it // NSKIP) - 1
        if it1 < NWMAX:
            VWX[it1, :] = VX[ISTX, ISTZ]
            VWZ[it1, :] = VZ[ISTX, ISTZ]

    # 進捗表示とスナップショット出力
    if it % NTDEC == 0:
        vxmax = np.max(VX)
        print(f"{it:5d}/{NTMAX:5d}: T={T:6.2f}[s] vxmax={vxmax:.3e}")
        
        # 空間データ一括計算
        i_idx, k_idx = np.arange(1, NX+1, NXD), np.arange(1, NZ+1, NZD)
        ii, kk = np.meshgrid(i_idx, k_idx, indexing='ij')
        
        # 物理量の計算 (div, rot)
        d_vx_x = (VX[ii, kk] - VX[ii-1, kk]) / DX
        d_vz_z = (VZ[ii, kk] - VZ[ii, kk-1]) / DZ
        d_vx_z = (VX[ii, kk+1] - VX[ii, kk]) / DZ
        d_vz_x = (VZ[ii+1, kk] - VZ[ii, kk]) / DX
        
        out_data = np.column_stack([
            ((ii-1)*DX).ravel(), ((kk-1)*DZ).ravel(),
            VX[ii, kk].ravel(), VZ[ii, kk].ravel(),
            (d_vx_x + d_vz_z).ravel(), (d_vz_x - d_vx_z).ravel()
        ])
        np.savetxt(f"{ONAME0}{it:05d}.out", out_data, fmt='%9.3f %9.3f %12.3e %12.3e %12.3e %12.3e')

# --- 波形データの最終保存 ---
print("波形データ保存中...")
time_axis = np.arange(1, NWMAX + 1) * DT * NSKIP
for ist in range(NST):
    wfname = f"{WNAME0}{int(ISTX[ist]*DX):04d}.dat"
    np.savetxt(wfname, np.column_stack([time_axis, VWX[:, ist], VWZ[:, ist]]), fmt='%9.3f %14.5e %14.5e')

print("完了")

800ステップ目（16秒後）の地震波動場を可視化する。領域の左側だけに$Q = 10$の強い減衰が仮定されている。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# --- 設定 ---
fact = 1e4 # scale factor
dt = 0.02
NX, NZ = 400 // 2, 400 // 2  # 格子数を空間的に間引いた

time_val = 16.0
step = int(time_val/DT)
step = int(step/100)*100
time_val = step*DT
filename = f"output/psv.h.{step:05d}.out"

# データ読み込み
out = pd.read_csv(filename, sep='\s+', names=('X','Z','VX','VZ','div','rot'))
    

fig, axs = plt.subplots(2, 2, figsize=(12, 10))
plt.rcParams.update({'font.size': 10})
    
# データ整形
## 震源からの距離へ
X = out['X'].values.reshape(NZ, NX)-I0*DX
Z = out['Z'].values.reshape(NZ, NX)-K0*DZ
## 波動場
VX = out['VX'].values.reshape(NZ, NX)
VZ = out['VZ'].values.reshape(NZ, NX)
VP = out['div'].values.reshape(NZ, NX)
VS = out['rot'].values.reshape(NZ, NX)
    
# 正規化
VX *= fact
VZ *= fact
VP *= fact*4
VS *= fact

# 左側：Vx
ax0 = axs[0,0]
im1 = ax0.pcolormesh(X, Z, VX, shading='auto', cmap='RdBu_r', vmin=-1, vmax=1)
label_text = fr"$V_X$ at {time_val:.2f} s"
ax0.text(0.95, 0.05, label_text,
        transform=ax0.transAxes,
        fontsize=12, va='bottom', ha='right', fontweight='regular')
ax0.set_ylabel('Z from source [km]',fontsize=14)

# 右側：Vz
ax1 = axs[0,1]
im2 = ax1.pcolormesh(X, Z, VZ, shading='auto', cmap='RdBu_r', vmin=-1, vmax=1)
label_text = fr"$V_Z$ at {time_val:.2f} s"
ax1.text(0.95, 0.05, label_text,
        transform=ax1.transAxes,
        fontsize=12, va='bottom', ha='right', fontweight='regular')
fig.colorbar(im2, ax=ax1, shrink=0.5).set_ticks([])

# 左側：div v
ax0 = axs[1,0]
im1 = ax0.pcolormesh(X, Z, VP, shading='auto', cmap='RdBu_r', vmin=-1, vmax=1)
label_text = fr"$\mathrm{{div}} \ \mathbf{{v}} \times 4$ at {time_val:.2f} s"
ax0.text(0.95, 0.05, label_text,
        transform=ax0.transAxes,
        fontsize=12, va='bottom', ha='right', fontweight='regular')
ax0.set_xlabel('X from source [km]',fontsize=14)
ax0.set_ylabel('Z from source [km]',fontsize=14)

# 右側：rot v
ax1 = axs[1,1]
im2 = ax1.pcolormesh(X, Z, VS, shading='auto', cmap='RdBu_r', vmin=-1, vmax=1)
label_text = fr"$\mathrm{{rot}} \ \mathbf{{v}}$ at {time_val:.2f} s"
ax1.text(0.95, 0.05, label_text,
        transform=ax1.transAxes,
        fontsize=12, va='bottom', ha='right', fontweight='regular')
ax1.set_xlabel('X from source [km]',fontsize=14)
fig.colorbar(im2, ax=ax1, shrink=0.5).set_ticks([])

for ax in axs.flatten():
    ax.set_xlim(-80, 80)
    ax.set_ylim(-80, 80)
    ax.set_aspect(1.0)
    ax.tick_params(direction="in", top=True, right=True, which='both')

plt.tight_layout()
plt.show()

以下では、波形ファイルを用いて$v_z$成分を描いています。まず初めに、震源からの距離が正、つまり右側に伝播する地震波形を描き、その後、左側（距離が負）へ伝播する地震波を描いています。

理論的には、2次元完全弾性体のシミュレーションでは震源距離を$r$とすると、遠地項について$1/\sqrt{r}$の幾何減衰が期待されます。

In [ ]:
import glob

dir_path = './output/'  # ファイルがあるディレクトリ
file_pattern = 'wav.h.*.dat'
files = sorted(glob.glob(os.path.join(dir_path, file_pattern)))
dist0 = 80.0

# attenuation function
A0 = 0.0016
VS = 3.500

# === プロット ===
plt.rcParams.update({'font.size': 11})
plt.rcParams['xtick.direction'] = 'in' 
plt.rcParams['ytick.direction'] = 'in' 
plt.rcParams["xtick.minor.visible"] = True  
plt.rcParams["ytick.minor.visible"] = True  

plt.figure(figsize=(5, 6))

for file in files:
    data = np.loadtxt(file)
    t  = data[:, 0]
    vz = data[:, 2]
    filename = os.path.basename(file)
    dist = int(filename.split('.')[2])-dist0

    if dist > 0:
        plt.plot(t, vz, label=f"{dist} [km]")
    
A = A0/np.sqrt(t*VS)

plt.plot(t, A, color='black',ls='--',label=r'$1 / \sqrt{r}$')

plt.legend()
plt.xlim(0,30)
plt.ylim(-0.0005, 0.0005)
plt.xlabel("Time [s]", fontsize=14)
plt.title(r"$Q = \infty $")
plt.tight_layout
plt.show()

In [ ]:
# === プロット ===
plt.rcParams.update({'font.size': 11})
plt.rcParams['xtick.direction'] = 'in' 
plt.rcParams['ytick.direction'] = 'in' 
plt.rcParams["xtick.minor.visible"] = True  
plt.rcParams["ytick.minor.visible"] = True  

plt.figure(figsize=(5, 6))

for file in files:
    data = np.loadtxt(file)
    t  = data[:, 0]
    vz = data[:, 2]
    filename = os.path.basename(file)
    dist = int(filename.split('.')[2])-dist0

    if dist < 0:
        plt.plot(t, -vz, label=f"{dist} [km]")
    
A = A0/np.sqrt(t*VS)

plt.plot(t, A, color='black',ls='--',label=r'$1 / \sqrt{r}$')

plt.legend()
plt.xlim(0,30)
plt.ylim(-0.0005, 0.0005)
plt.xlabel("Time [s]", fontsize=14)
plt.title("$Q = 10$")
plt.tight_layout
plt.show()